# mech-sim Colab Workflow

This notebook keeps the existing `mech-sim` workflow intact: it rebuilds the Rust crate from scratch, reruns the baseline scenarios and sweeps, regenerates figures from exported CSV/JSON artifacts, writes a PDF report, and produces a download-ready zip bundle.

The upgrade added here is a **reference analysis layer**, not a replacement for the Rust simulator.

## Why the reduced multi-timescale model matters

The paper's central claim is architectural: gigawatt-class terrestrial legged vehicles are governed by incompatible timescales across continuous generation, pulse storage, local actuation, and thermal rejection. The argument is strongest when it can be shown in a reduced-order model that is simple enough to inspect and rich enough to couple the relevant state constraints.

The reduced state used by the crate and mirrored in the notebook is:

- `Ep(t)`: pulse-layer stored energy,
- `T(t)`: aggregate thermal state,
- `y(t)`: reduced maneuver response,
- `v(t) = dy/dt`: reduced maneuver-response rate,
- `Elocal_i(t)`: four limb-local energy buffers in the full Rust model.

At architecture level, the coupled dynamics are:

```text
dEp/dt = eta_c * Pc - Ptransfer_total - Pl(Ep, T)
Ct * dT/dt = Qg(Pdelivered, Ptransfer_total, Pl) - Qr(T)
M * y_ddot + D(T) * y_dot + K(y, T) * y = G(Ep, T) * u + d(t)
dElocal_i/dt = Ptransfer_i - Pdraw_i - Plocal_loss
```

The notebook adds a **SciPy reference implementation** for the core reduced state `(Ep, T, y, v)` using `solve_ivp`. This serves three purposes:

- it provides a transparent mathematical baseline alongside the Rust crate outputs,
- it cross-checks trajectory shape and event timing against exported Rust CSV data,
- it enables rapid interactive parameter exploration without touching the crate implementation.

## Interpreting `y`, mechanical work, and recharge timing

The state `y` is a **reduced-order maneuver-response proxy**. It is intentionally coupled to the energy and thermal states, but it is not intended to imply literal full-body mech displacement unless that interpretation is independently justified. The paper-support figures and report therefore label `y` as a reduced maneuver response or reduced mechanical output, not as direct full-body motion.

For the same reason, delivered mechanical work should be read as **architecture-level evidence** for how much maneuver authority survives energy and thermal constraints. Small work values in this reduced model do not invalidate the pulse-layer, recharge, thermal, or authority-collapse conclusions; they mainly show that this crate validates the layered architecture more strongly than it validates full rigid-body maneuver realism.

Recharge timing must also be read in context. A burst scenario can show a recharge tail of tens of seconds because it only refills the energy actually depleted by that burst. The dedicated `recharge` case is a larger refill experiment centered on a near-empty 3 GJ store at `Pc = 50 MW`, so an approximately 60 second refill is not contradictory. The comparison is partial refill versus larger full-reserve refill depth.

## Admissible operating region and state-constrained maneuver authority

The paper's strongest formal idea is that maneuver authority is **state-constrained**. The same command input is not equally available everywhere in state space because both pulse energy and thermal state degrade effective gain.

In notebook form, that idea is visualized with an authority map over `(Ep, T)` using a normalized gain surface:

```text
G_eff(Ep, T) = G0 * s_energy(Ep) * s_thermal(T)
authority_norm = G_eff / G0
```

The admissible region is then partitioned into:

- nominal,
- degraded,
- energy-limited,
- thermal-limited,
- inadmissible.

That is the missing bridge between a purely symbolic reduced-order model and a credible proof-of-life validation layer for the paper.

## What this notebook validates

This notebook validates reduced-order coherence of:

- pulse depletion vs recharge,
- aggregate thermal accumulation and rejection,
- reduced maneuver response under constrained authority,
- state-constrained maneuver authority,
- sensitivity to charging efficiency, induced-power burden, thermal admissibility, and burst demand.

It does **not** claim full vehicle validation. It remains intentionally reduced-order and does not replace multibody, terrain-contact, or high-fidelity hydraulic simulation.


In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys
import zipfile

def run(cmd, cwd=None, env=None):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

run(['apt-get', 'update'])
run([
    'apt-get',
    'install',
    '-y',
    'git',
    'curl',
    'build-essential',
    'pkg-config',
    'libfontconfig1-dev',
    'libfreetype6-dev',
])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'numpy', 'scipy', 'ipywidgets'])

cargo_bin = pathlib.Path.home() / '.cargo' / 'bin'
if shutil.which('cargo') is None:
    run(['bash', '-lc', 'curl https://sh.rustup.rs -sSf | sh -s -- -y'])
os.environ['PATH'] = f"{cargo_bin}:{os.environ['PATH']}"
run(['cargo', '--version'])
run(['rustc', '--version'])

repo_url = os.environ.get('MECH_REPO_URL', 'https://github.com/infinityabundance/mech.git')
workspace = pathlib.Path('/content/mech')
if workspace.exists() and not (workspace / '.git').exists():
    shutil.rmtree(workspace)
if not workspace.exists():
    run(['git', 'clone', repo_url, str(workspace)])

output_root = workspace / 'output-mech-sim'
shutil.rmtree(output_root, ignore_errors=True)
workspace

In [ ]:
def run_and_capture(args):
    before = {p.name for p in output_root.iterdir()} if output_root.exists() else set()
    run(['cargo', 'run', '-p', 'mech-sim', '--', *args], cwd=workspace)
    after = sorted([p for p in output_root.iterdir() if p.name not in before])
    if len(after) != 1:
        raise RuntimeError(f'Expected exactly one new output directory, found: {after}')
    return after[0]

run(['cargo', 'build', '-p', 'mech-sim'], cwd=workspace)

burst_dir = run_and_capture(['scenario', 'burst'])
hover_dir = run_and_capture(['scenario', 'hover'])
stress_dir = run_and_capture(['scenario', 'stress'])
sweep_dir = run_and_capture(['sweep', 'baseline'])
thermal_duty_dir = run_and_capture(['sweep', 'thermal-duty-matrix'])
allocation_dir = run_and_capture(['sweep', 'limb-allocation-comparison'])

run_roots = {
    'burst': burst_dir,
    'hover': hover_dir,
    'stress': stress_dir,
    'sweep': sweep_dir,
    'thermal_duty': thermal_duty_dir,
    'allocation': allocation_dir,
}

# Notebook artifacts are kept inside the same timestamped output tree as the burst run.
analysis_dir = burst_dir / 'notebook_artifacts'
figure_dir = analysis_dir / 'figures'
data_dir = analysis_dir / 'data'
report_dir = analysis_dir / 'reports'
for directory in [analysis_dir, figure_dir, data_dir, report_dir]:
    directory.mkdir(parents=True, exist_ok=True)

(data_dir / 'run_roots.json').write_text(json.dumps({key: str(value) for key, value in run_roots.items()}, indent=2))
{'run_roots': run_roots, 'analysis_dir': analysis_dir}


## Reference SciPy model and interactive exploration

The Rust crate remains the authoritative simulator. The code below adds a notebook-only reference implementation for the reduced state `(Ep, T, y, v)` using `scipy.integrate.solve_ivp`.

It is intentionally labeled as a reference layer because:

- it mirrors the reduced-order equations already encoded by the crate,
- it does not replace the crate's local-buffer detail,
- it is primarily for cross-checking, sensitivity exploration, and figure generation.

The interactive controls expose the paper's main sensitivities:

- `eta_c`: recharge efficiency,
- `thrust_area`: a notebook-side induced-power sensitivity knob,
- `T_max`: thermal admissibility limit,
- `qr_gain`: thermal rejection coefficient,
- `Pc_MW`: continuous charging power,
- `Ep0_GJ`: initial pulse-layer energy,
- `burst_MW`: burst demand scale.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.integrate import solve_ivp
import ipywidgets as widgets

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 160,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'font.size': 11,
})

burst_ts = pd.read_csv(burst_dir / 'time_series.csv')
hover_ts = pd.read_csv(hover_dir / 'time_series.csv')
stress_ts = pd.read_csv(stress_dir / 'time_series.csv')
stress_limb = pd.read_csv(stress_dir / 'limb_buffers.csv')
sweep_summary = pd.read_csv(sweep_dir / 'sweep_summary.csv')
thermal_duty_summary = pd.read_csv(thermal_duty_dir / 'thermal_duty_matrix.csv')
allocation_summary = pd.read_csv(allocation_dir / 'limb_allocation_comparison.csv')

burst_params = json.loads((burst_dir / 'params.json').read_text())
hover_params = json.loads((hover_dir / 'params.json').read_text())
stress_params = json.loads((stress_dir / 'params.json').read_text())
burst_summary = json.loads((burst_dir / 'summary.json').read_text())
hover_summary = json.loads((hover_dir / 'summary.json').read_text())
stress_summary = json.loads((stress_dir / 'summary.json').read_text())
burst_stability_summary = json.loads((burst_dir / 'stability_summary.json').read_text())
hover_stability_summary = json.loads((hover_dir / 'stability_summary.json').read_text())
stress_stability_summary = json.loads((stress_dir / 'stability_summary.json').read_text())
burst_figure_metadata = json.loads((burst_dir / 'figure_metadata.json').read_text())
hover_figure_metadata = json.loads((hover_dir / 'figure_metadata.json').read_text())
stress_figure_metadata = json.loads((stress_dir / 'figure_metadata.json').read_text())

def save_figure_pair(fig, stem, directory=figure_dir):
    png_path = directory / f'{stem}.png'
    pdf_path = directory / f'{stem}.pdf'
    fig.savefig(png_path, dpi=220, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    return {'png': png_path, 'pdf': pdf_path}

def save_dataframe(df, name, directory=data_dir):
    path = directory / name
    df.to_csv(path, index=False)
    return path

def write_json(name, payload, directory=data_dir):
    path = directory / name
    path.write_text(json.dumps(payload, indent=2))
    return path

def smoothstep01(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)

def extract_burst_windows(scenario):
    return [
        (segment['start_s'], segment['end_s'], segment['label'])
        for segment in scenario.get('segments', [])
        if segment.get('demand_fraction', 0.0) > 0.05
    ]

def shade_burst_windows(ax, scenario, color='#f6c27a', alpha=0.18):
    for start_s, end_s, _ in extract_burst_windows(scenario):
        ax.axvspan(start_s, end_s, color=color, alpha=alpha, lw=0)

def scenario_command_at(t, scenario):
    command = scenario.get('idle_command', 0.0)
    label = 'idle'
    disturbance = 0.0
    for segment in scenario.get('segments', []):
        if segment['start_s'] <= t < segment['end_s'] and segment['demand_fraction'] >= command:
            command = segment['demand_fraction']
            label = segment['label']
            disturbance = segment.get('disturbance_n', 0.0)
    return command, label, disturbance

def make_reference_scenario(duration_s=60.0):
    return {
        'name': 'interactive-duty-cycle',
        'description': 'Notebook-only repeated burst/coast/recharge profile for interactive exploration.',
        'idle_command': 0.02,
        'segments': [
            {'label': 'burst_a', 'start_s': 5.0, 'end_s': 6.0, 'demand_fraction': 1.0, 'disturbance_n': 0.0},
            {'label': 'burst_b', 'start_s': 17.0, 'end_s': 18.0, 'demand_fraction': 0.92, 'disturbance_n': 0.0},
            {'label': 'burst_c', 'start_s': 29.0, 'end_s': 30.0, 'demand_fraction': 0.96, 'disturbance_n': 0.0},
            {'label': 'burst_d', 'start_s': 41.0, 'end_s': 42.0, 'demand_fraction': 0.90, 'disturbance_n': 0.0},
        ],
        'duration_s': duration_s,
    }

def make_reference_params(rust_config):
    model = rust_config['model']
    return {
        'ambient_temperature_k': model['ambient_temperature_k'],
        'thermal_initial_k': model['thermal_initial_k'],
        'thermal_capacity_j_per_k': model['thermal_capacity_j_per_k'],
        'thermal_soft_limit_k': model['thermal_soft_limit_k'],
        'thermal_limit_k': model['thermal_limit_k'],
        'thermal_rejection_w_per_k': model['thermal_rejection_w_per_k'],
        'thermal_rejection_quadratic_w_per_k2': model['thermal_rejection_quadratic_w_per_k2'],
        'recharge_efficiency': model['recharge_efficiency'],
        'continuous_power_w': model['continuous_power_w'],
        'pulse_energy_max_j': model['pulse_energy_max_j'],
        'pulse_energy_initial_j': model['pulse_energy_initial_j'],
        'pulse_energy_min_j': model['pulse_energy_min_j'],
        'low_energy_threshold_j': model['low_energy_threshold_j'],
        'actuator_peak_power_w': model['actuator_peak_power_w'],
        'actuator_idle_power_w': model['actuator_idle_power_w'],
        'actuator_velocity_power_coeff_w_per_mps': model['actuator_velocity_power_coeff_w_per_mps'],
        'actuator_position_power_coeff_w_per_m': model['actuator_position_power_coeff_w_per_m'],
        'actuator_heat_fraction': model['actuator_heat_fraction'],
        'transfer_heat_fraction': model['transfer_heat_fraction'],
        'loss_idle_w': model['loss_idle_w'],
        'loss_storage_coeff_w': model['loss_storage_coeff_w'],
        'loss_thermal_coeff_w_per_k': model['loss_thermal_coeff_w_per_k'],
        'mechanical_mass_kg': model['mechanical_mass_kg'],
        'damping_n_s_per_m': model['damping_n_s_per_m'],
        'stiffness_n_per_m': model['stiffness_n_per_m'],
        'reference_actuator_force_n': model['reference_actuator_force_n'],
        'damping_temp_coeff': model['damping_temp_coeff'],
        'stiffness_temp_softening': model['stiffness_temp_softening'],
        'stiffness_position_coeff': model['stiffness_position_coeff'],
        'min_gain_fraction': model['min_gain_fraction'],
        'energy_gain_soft_zone_j': model['energy_gain_soft_zone_j'],
        'thermal_gain_soft_zone_k': model['thermal_gain_soft_zone_k'],
        'max_displacement_m': model['max_displacement_m'],
        'max_velocity_m_per_s': model['max_velocity_m_per_s'],
        'thrust_area_m2': 16.0,
        'reference_thrust_area_m2': 16.0,
        'hover_penalty_coeff_w': 7.5e7,
        'readiness_fraction': 0.82,
    }

def reference_energy_factor(ep_j, params):
    normalized = (ep_j - params['low_energy_threshold_j']) / max(params['energy_gain_soft_zone_j'], 1.0)
    return params['min_gain_fraction'] + (1.0 - params['min_gain_fraction']) * smoothstep01(normalized)

def reference_thermal_factor(temperature_k, params):
    normalized = (params['thermal_limit_k'] - temperature_k) / max(params['thermal_gain_soft_zone_k'], 1.0)
    return params['min_gain_fraction'] + (1.0 - params['min_gain_fraction']) * smoothstep01(normalized)

def actuator_power_reference(command_fraction, y_m, v_mps, params):
    area_scale = np.sqrt(params['reference_thrust_area_m2'] / max(params['thrust_area_m2'], 1.0e-6))
    induced_power = params['hover_penalty_coeff_w'] * max(command_fraction, 0.0) ** 1.5 * area_scale
    return (
        params['actuator_idle_power_w']
        + params['actuator_peak_power_w'] * max(command_fraction, 0.0) ** 1.15
        + params['actuator_velocity_power_coeff_w_per_mps'] * abs(v_mps)
        + params['actuator_position_power_coeff_w_per_m'] * abs(y_m)
        + induced_power
    )

def parasitic_loss_reference(ep_j, temperature_k, params):
    soc = np.clip(ep_j / max(params['pulse_energy_max_j'], 1.0), 0.0, 1.5)
    thermal_excess = max(temperature_k - params['ambient_temperature_k'], 0.0)
    return (
        params['loss_idle_w']
        + params['loss_storage_coeff_w'] * soc * soc
        + params['loss_thermal_coeff_w_per_k'] * thermal_excess
    )

def heat_generation_reference(power_w, parasitic_loss_w, params):
    return (
        params['actuator_heat_fraction'] * power_w
        + params['transfer_heat_fraction'] * power_w
        + parasitic_loss_w
    )

def heat_rejection_reference(temperature_k, params):
    delta_k = max(temperature_k - params['ambient_temperature_k'], 0.0)
    return (
        params['thermal_rejection_w_per_k'] * delta_k
        + params['thermal_rejection_quadratic_w_per_k2'] * delta_k * delta_k
    )

def mechanical_terms(y_m, v_mps, ep_j, temperature_k, command_fraction, disturbance_n, params):
    energy_factor = reference_energy_factor(ep_j, params)
    thermal_factor = reference_thermal_factor(temperature_k, params)
    temp_fraction = np.clip(
        (temperature_k - params['ambient_temperature_k']) / max(params['thermal_limit_k'] - params['ambient_temperature_k'], 1.0),
        0.0,
        1.5,
    )
    damping = params['damping_n_s_per_m'] * (1.0 + params['damping_temp_coeff'] * temp_fraction)
    stiffness = (
        params['stiffness_n_per_m']
        * max(1.0 - params['stiffness_temp_softening'] * temp_fraction, 0.35)
        * (1.0 + params['stiffness_position_coeff'] * y_m * y_m)
    )
    gain = params['reference_actuator_force_n'] * energy_factor * thermal_factor
    force_n = gain * command_fraction + disturbance_n
    accel = (force_n - damping * v_mps - stiffness * y_m) / max(params['mechanical_mass_kg'], 1.0)
    return {
        'gain': gain,
        'damping': damping,
        'stiffness': stiffness,
        'accel': accel,
        'authority_norm': energy_factor * thermal_factor,
    }

def rhs(t, state, params, scenario):
    ep_j, temperature_k, y_m, v_mps = state
    command_fraction, _, disturbance_n = scenario_command_at(t, scenario)
    power_w = actuator_power_reference(command_fraction, y_m, v_mps, params)
    parasitic_loss_w = parasitic_loss_reference(ep_j, temperature_k, params)
    thermal_terms = mechanical_terms(y_m, v_mps, ep_j, temperature_k, command_fraction, disturbance_n, params)
    dep_dt = params['recharge_efficiency'] * params['continuous_power_w'] - power_w - parasitic_loss_w
    dtemp_dt = (
        heat_generation_reference(power_w, parasitic_loss_w, params)
        - heat_rejection_reference(temperature_k, params)
    ) / max(params['thermal_capacity_j_per_k'], 1.0)
    return [dep_dt, dtemp_dt, v_mps, thermal_terms['accel']]

def classify_region(ep_j, temperature_k, authority_norm, params):
    low_energy = ep_j < params['low_energy_threshold_j']
    high_temperature = temperature_k >= params['thermal_limit_k']
    if low_energy and high_temperature:
        return 'inadmissible'
    if low_energy:
        return 'energy-limited'
    if high_temperature:
        return 'thermal-limited'
    if authority_norm < 0.55:
        return 'degraded'
    return 'nominal'

def run_reference(params, scenario, duration_s=None, dt_s=0.05):
    duration_s = duration_s if duration_s is not None else scenario.get('duration_s', 60.0)
    t_eval = np.arange(0.0, duration_s + dt_s, dt_s)
    solution = solve_ivp(
        lambda t, state: rhs(t, state, params, scenario),
        (0.0, duration_s),
        [
            params['pulse_energy_initial_j'],
            params['thermal_initial_k'],
            0.0,
            0.0,
        ],
        t_eval=t_eval,
        max_step=dt_s,
        rtol=1e-6,
        atol=1e-8,
    )
    if not solution.success:
        raise RuntimeError(solution.message)

    records = []
    for time_s, ep_j, temperature_k, y_m, v_mps in zip(solution.t, *solution.y):
        command_fraction, active_segment, disturbance_n = scenario_command_at(time_s, scenario)
        power_w = actuator_power_reference(command_fraction, y_m, v_mps, params)
        parasitic_loss_w = parasitic_loss_reference(ep_j, temperature_k, params)
        mech = mechanical_terms(y_m, v_mps, ep_j, temperature_k, command_fraction, disturbance_n, params)
        records.append({
            'time_s': time_s,
            'ep_j': max(ep_j, 0.0),
            'ep_gj': max(ep_j, 0.0) / 1.0e9,
            'temperature_k': max(temperature_k, params['ambient_temperature_k']),
            'temperature_c': max(temperature_k, params['ambient_temperature_k']) - 273.15,
            'y_m': np.clip(y_m, -params['max_displacement_m'], params['max_displacement_m']),
            'v_mps': np.clip(v_mps, -params['max_velocity_m_per_s'], params['max_velocity_m_per_s']),
            'command_fraction': command_fraction,
            'active_segment': active_segment,
            'requested_actuator_power_w': power_w,
            'parasitic_loss_w': parasitic_loss_w,
            'gain': mech['gain'],
            'damping': mech['damping'],
            'stiffness': mech['stiffness'],
            'authority_norm': mech['authority_norm'],
            'low_energy': ep_j < params['low_energy_threshold_j'],
            'high_temperature': temperature_k >= params['thermal_limit_k'],
            'readiness_threshold_j': params['readiness_fraction'] * params['pulse_energy_max_j'],
        })
    df = pd.DataFrame.from_records(records)
    df['readiness'] = df['ep_j'] >= df['readiness_threshold_j']
    df['degraded'] = df['authority_norm'] < 0.55
    df['region'] = [classify_region(ep, temp, authority, params) for ep, temp, authority in zip(df['ep_j'], df['temperature_k'], df['authority_norm'])]
    df['energy_event'] = df['low_energy'] & ~df['low_energy'].shift(fill_value=False)
    df['thermal_event'] = df['high_temperature'] & ~df['high_temperature'].shift(fill_value=False)
    df['degraded_event'] = df['degraded'] & ~df['degraded'].shift(fill_value=False)
    return df

def compute_duty_cycle_metrics(df, scenario, params):
    readiness_threshold_j = params['readiness_fraction'] * params['pulse_energy_max_j']
    cycle_rows = []
    for start_s, end_s, label in extract_burst_windows(scenario):
        post_burst = df[df['time_s'] >= end_s].copy()
        recovered = post_burst[post_burst['ep_j'] >= readiness_threshold_j]
        recovered_at_s = float(recovered.iloc[0]['time_s']) if not recovered.empty else np.nan
        cycle_rows.append({
            'label': label,
            'burst_start_s': start_s,
            'burst_end_s': end_s,
            'recovered_at_s': recovered_at_s,
            'recharge_time_s': recovered_at_s - end_s if np.isfinite(recovered_at_s) else np.nan,
        })
    cycle_df = pd.DataFrame(cycle_rows)
    dt_s = float(np.median(np.diff(df['time_s']))) if len(df) > 1 else 0.05
    energy_deficit_area_gj_s = np.trapz(np.clip((readiness_threshold_j - df['ep_j']) / 1.0e9, 0.0, None), df['time_s'])
    duty_cycle_fraction = float((df['command_fraction'] > 0.2).sum() * dt_s / max(df['time_s'].iloc[-1], 1.0e-6))
    summary = {
        'readiness_threshold_gj': readiness_threshold_j / 1.0e9,
        'mean_recharge_time_s': float(cycle_df['recharge_time_s'].dropna().mean()) if not cycle_df.empty else np.nan,
        'energy_deficit_area_gj_s': float(energy_deficit_area_gj_s),
        'duty_cycle_fraction': float(duty_cycle_fraction),
        'first_low_energy_s': float(df.loc[df['energy_event'], 'time_s'].iloc[0]) if df['energy_event'].any() else np.nan,
        'first_thermal_limit_s': float(df.loc[df['thermal_event'], 'time_s'].iloc[0]) if df['thermal_event'].any() else np.nan,
    }
    return cycle_df, summary

def mark_events(ax, df, y_column, event_column, label, color):
    event_rows = df[df[event_column]]
    if not event_rows.empty:
        ax.scatter(event_rows['time_s'], event_rows[y_column], color=color, s=28, zorder=5, label=label)

def make_coupled_figure(df, params, scenario, title):
    caption = 'Coupled evolution of pulse-layer energy, aggregate thermal state, mechanical response, and actuator demand under a representative burst–recharge maneuver profile.'
    fig, axes = plt.subplots(4, 1, figsize=(11.5, 10.0), sharex=True)
    series = [
        ('ep_gj', 'Ep [GJ]', '#1f77b4'),
        ('temperature_c', 'T [C]', '#d62728'),
        ('y_m', 'y [m]', '#9467bd'),
        ('requested_actuator_power_w', 'Pa [MW]', '#ff7f0e'),
    ]
    for ax, (column, ylabel, color) in zip(axes, series):
        y_values = df[column] / 1.0e6 if column == 'requested_actuator_power_w' else df[column]
        ax.plot(df['time_s'], y_values, color=color, lw=2.4)
        shade_burst_windows(ax, scenario)
        ax.set_ylabel(ylabel)
        mark_events(ax, df.assign(requested_actuator_power_w_mw=df['requested_actuator_power_w'] / 1.0e6), column if column != 'requested_actuator_power_w' else 'requested_actuator_power_w_mw', 'energy_event', 'Energy limit', '#b2182b')
        mark_events(ax, df.assign(requested_actuator_power_w_mw=df['requested_actuator_power_w'] / 1.0e6), column if column != 'requested_actuator_power_w' else 'requested_actuator_power_w_mw', 'thermal_event', 'Thermal limit', '#2166ac')
    axes[0].axhline(params['low_energy_threshold_j'] / 1.0e9, color='#555555', ls='--', lw=1.1)
    axes[1].axhline(params['thermal_limit_k'] - 273.15, color='#555555', ls='--', lw=1.1)
    axes[-1].set_xlabel('Time [s]')
    axes[0].set_title(title)
    fig.text(0.01, 0.01, caption, fontsize=10)
    fig.tight_layout(rect=[0.0, 0.03, 1.0, 0.98])
    return fig, caption

def make_recharge_figure(df, params, scenario, title):
    cycle_df, duty_summary = compute_duty_cycle_metrics(df, scenario, params)
    caption = 'Duty-cycle behavior of the layered energy architecture under repeated burst loading, showing pulse depletion, recharge recovery, and readiness thresholds.'
    fig, axes = plt.subplots(2, 1, figsize=(11.5, 7.4), sharex=True)
    readiness_threshold_gj = duty_summary['readiness_threshold_gj']
    axes[0].plot(df['time_s'], df['ep_gj'], color='#1f77b4', lw=2.4)
    axes[0].axhline(readiness_threshold_gj, color='#444444', ls='--', lw=1.2, label='Readiness threshold')
    shade_burst_windows(axes[0], scenario)
    degraded_rows = df[df['degraded'] | df['low_energy'] | df['high_temperature']]
    if not degraded_rows.empty:
        axes[0].scatter(degraded_rows['time_s'], degraded_rows['ep_gj'], color='#b2182b', s=14, alpha=0.6, label='Degraded / failed')
    for _, row in cycle_df.iterrows():
        if np.isfinite(row['recovered_at_s']):
            axes[0].annotate(
                f"{row['recharge_time_s']:.1f} s",
                xy=(row['recovered_at_s'], readiness_threshold_gj),
                xytext=(row['recovered_at_s'] + 1.0, readiness_threshold_gj + 0.12),
                arrowprops={'arrowstyle': '->', 'lw': 0.8},
                fontsize=9,
            )
    axes[0].set_ylabel('Ep [GJ]')
    axes[0].set_title(title)
    axes[0].legend(loc='upper right')

    axes[1].plot(df['time_s'], df['command_fraction'], color='#ff7f0e', lw=2.0, label='Command fraction')
    axes[1].fill_between(df['time_s'], 0.0, df['command_fraction'], color='#ff7f0e', alpha=0.18)
    shade_burst_windows(axes[1], scenario)
    axes[1].set_ylabel('u [-]')
    axes[1].set_xlabel('Time [s]')
    axes[1].text(
        0.01,
        0.88,
        f"Mean recharge time: {duty_summary['mean_recharge_time_s']:.2f} s\nEnergy deficit area: {duty_summary['energy_deficit_area_gj_s']:.2f} GJ*s\nDuty-cycle fraction: {duty_summary['duty_cycle_fraction']:.2f}",
        transform=axes[1].transAxes,
        fontsize=10,
        bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '#cccccc'},
    )
    fig.text(0.01, 0.01, caption, fontsize=10)
    fig.tight_layout(rect=[0.0, 0.03, 1.0, 0.98])
    return fig, caption, cycle_df, duty_summary

def make_authority_map(params, title):
    caption = 'State-constrained maneuver authority over the pulse-energy/thermal-state plane, illustrating degradation and inadmissible operating regions. This is the visual counterpart of the reduced map M(x, u) = G(Ep, T) u over the admissible operating region.'
    ep_grid = np.linspace(max(params['pulse_energy_min_j'], 1.0e6), params['pulse_energy_max_j'], 140)
    temp_grid = np.linspace(params['ambient_temperature_k'], params['thermal_limit_k'] + 12.0, 140)
    ep_mesh, temp_mesh = np.meshgrid(ep_grid, temp_grid)
    authority_mesh = reference_energy_factor(ep_mesh, params) * reference_thermal_factor(temp_mesh, params)
    regions = np.empty_like(authority_mesh, dtype=object)
    for row in range(authority_mesh.shape[0]):
        for col in range(authority_mesh.shape[1]):
            regions[row, col] = classify_region(ep_mesh[row, col], temp_mesh[row, col], authority_mesh[row, col], params)
    authority_df = pd.DataFrame({
        'ep_j': ep_mesh.ravel(),
        'ep_gj': ep_mesh.ravel() / 1.0e9,
        'temperature_k': temp_mesh.ravel(),
        'temperature_c': temp_mesh.ravel() - 273.15,
        'authority_norm': authority_mesh.ravel(),
        'region': regions.ravel(),
    })

    fig, ax = plt.subplots(figsize=(10.2, 7.0))
    heatmap = ax.pcolormesh(ep_mesh / 1.0e9, temp_mesh - 273.15, authority_mesh, shading='auto', cmap='viridis')
    contour_levels = [0.35, 0.55, 0.75, 0.9]
    contours = ax.contour(ep_mesh / 1.0e9, temp_mesh - 273.15, authority_mesh, levels=contour_levels, colors='white', linewidths=0.8)
    ax.clabel(contours, inline=True, fontsize=8, fmt='%.2f')
    ax.axvline(params['low_energy_threshold_j'] / 1.0e9, color='#ffeda0', ls='--', lw=1.2)
    ax.axhline(params['thermal_limit_k'] - 273.15, color='#f03b20', ls='--', lw=1.2)
    ax.set_title(title)
    ax.set_xlabel('Pulse-layer energy Ep [GJ]')
    ax.set_ylabel('Aggregate thermal state T [C]')
    colorbar = fig.colorbar(heatmap, ax=ax)
    colorbar.set_label('Normalized authority [-]')
    ax.text(params['pulse_energy_max_j'] * 0.78 / 1.0e9, params['ambient_temperature_k'] - 273.15 + 6.0, 'nominal', color='white', fontsize=10)
    ax.text(params['pulse_energy_max_j'] * 0.55 / 1.0e9, params['thermal_limit_k'] - 273.15 - 6.0, 'degraded', color='white', fontsize=10)
    ax.text(params['low_energy_threshold_j'] * 0.55 / 1.0e9, params['ambient_temperature_k'] - 273.15 + 8.0, 'energy-limited', color='black', fontsize=9)
    ax.text(params['pulse_energy_max_j'] * 0.78 / 1.0e9, params['thermal_limit_k'] - 273.15 + 3.0, 'thermal-limited', color='black', fontsize=9)
    ax.text(params['low_energy_threshold_j'] * 0.55 / 1.0e9, params['thermal_limit_k'] - 273.15 + 3.0, 'inadmissible', color='black', fontsize=9)
    fig.text(0.01, 0.01, caption, fontsize=10)
    fig.tight_layout(rect=[0.0, 0.03, 1.0, 0.98])
    return fig, caption, authority_df

def compare_to_rust(reference_df, rust_df):
    aligned = pd.DataFrame({'time_s': reference_df['time_s']})
    for ref_column, rust_column in [
        ('ep_j', 'ep_j'),
        ('temperature_k', 'temperature_k'),
        ('y_m', 'y_m'),
        ('requested_actuator_power_w', 'requested_actuator_power_w'),
        ('authority_norm', 'authority_utilization'),
    ]:
        rust_values = np.interp(reference_df['time_s'], rust_df['time_s'], rust_df[rust_column])
        aligned[f'rust_{ref_column}'] = rust_values
        aligned[f'scipy_{ref_column}'] = reference_df[ref_column].to_numpy()
        aligned[f'abs_err_{ref_column}'] = np.abs(aligned[f'scipy_{ref_column}'] - aligned[f'rust_{ref_column}'])
    metrics = {
        'ep_mae_gj': float(aligned['abs_err_ep_j'].mean() / 1.0e9),
        'ep_max_abs_err_gj': float(aligned['abs_err_ep_j'].max() / 1.0e9),
        'temperature_mae_k': float(aligned['abs_err_temperature_k'].mean()),
        'temperature_max_abs_err_k': float(aligned['abs_err_temperature_k'].max()),
        'y_mae_m': float(aligned['abs_err_y_m'].mean()),
        'power_mae_mw': float(aligned['abs_err_requested_actuator_power_w'].mean() / 1.0e6),
    }
    return aligned, metrics

def make_comparison_figure(reference_df, rust_df, metrics, title):
    fig, axes = plt.subplots(4, 1, figsize=(11.5, 10.0), sharex=True)
    pairs = [
        ('ep_gj', 'ep_gj', 'Ep [GJ]'),
        ('temperature_c', 'temperature_c', 'T [C]'),
        ('y_m', 'y_m', 'Reduced y [proxy m]'),
        ('requested_actuator_power_w', 'requested_actuator_power_w', 'Pa [MW]'),
    ]
    for ax, (ref_col, rust_col, ylabel) in zip(axes, pairs):
        ref_values = reference_df[ref_col] / 1.0e6 if ref_col == 'requested_actuator_power_w' else reference_df[ref_col]
        rust_values = rust_df[rust_col] / 1.0e6 if rust_col == 'requested_actuator_power_w' else rust_df[rust_col]
        ax.plot(reference_df['time_s'], ref_values, lw=2.3, color='#1f77b4', label='SciPy reference')
        ax.plot(rust_df['time_s'], rust_values, lw=1.7, color='#d62728', ls='--', label='Rust crate')
        ax.set_ylabel(ylabel)
    axes[0].set_title(title)
    axes[-1].set_xlabel('Time [s]')
    axes[0].legend(loc='upper right')
    axes[0].text(
        0.01,
        0.06,
        f"Ep MAE: {metrics['ep_mae_gj']:.4f} GJ | T MAE: {metrics['temperature_mae_k']:.4f} K | y MAE: {metrics['y_mae_m']:.4f} m | Pa MAE: {metrics['power_mae_mw']:.2f} MW",
        transform=axes[0].transAxes,
        fontsize=10,
        bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '#cccccc'},
    )
    fig.tight_layout()
    return fig

base_reference_params = make_reference_params(burst_params)
interactive_scenario = make_reference_scenario()


## Interactive slider surface

The widgets below rerun the SciPy reference model and refresh the three publication-ready figures:

1. coupled evolution,
2. recharge duty-cycle behavior,
3. authority map.

The authority map is the visual counterpart of the reduced mathematical statement

`M(x, u) = G(Ep, T) u`

in the sense that it shows how maneuver authority changes over the `(Ep, T)` plane and how degraded or inadmissible regions emerge relative to the admissible operating region.

The current slider values and derived metrics are written to the notebook artifact directory inside the burst timestamped run root, so the interactive state remains exportable and reproducible.


In [ ]:
interactive_output = widgets.Output()
interactive_state = {}

eta_slider = widgets.FloatSlider(description='eta_c', min=0.20, max=0.95, step=0.01, value=base_reference_params['recharge_efficiency'], readout_format='.2f', continuous_update=False)
area_slider = widgets.FloatSlider(description='thrust_area', min=8.0, max=30.0, step=0.5, value=16.0, readout_format='.1f', continuous_update=False)
tmax_slider = widgets.FloatSlider(description='T_max [K]', min=320.0, max=360.0, step=1.0, value=base_reference_params['thermal_limit_k'], readout_format='.1f', continuous_update=False)
qr_slider = widgets.FloatSlider(description='qr_gain [MW/K]', min=1.0, max=8.0, step=0.1, value=base_reference_params['thermal_rejection_w_per_k'] / 1.0e6, readout_format='.1f', continuous_update=False)
pc_slider = widgets.FloatSlider(description='Pc_MW', min=40.0, max=60.0, step=1.0, value=base_reference_params['continuous_power_w'] / 1.0e6, readout_format='.0f', continuous_update=False)
ep0_slider = widgets.FloatSlider(description='Ep0_GJ', min=1.0, max=5.0, step=0.1, value=min(5.0, base_reference_params['pulse_energy_initial_j'] / 1.0e9), readout_format='.1f', continuous_update=False)
burst_slider = widgets.FloatSlider(description='burst_MW', min=800.0, max=1000.0, step=10.0, value=base_reference_params['actuator_peak_power_w'] / 1.0e6, readout_format='.0f', continuous_update=False)

def render_interactive(*_):
    params = dict(base_reference_params)
    params['recharge_efficiency'] = eta_slider.value
    params['thrust_area_m2'] = area_slider.value
    params['thermal_limit_k'] = tmax_slider.value
    params['thermal_soft_limit_k'] = max(tmax_slider.value - 12.0, params['ambient_temperature_k'] + 5.0)
    params['thermal_rejection_w_per_k'] = qr_slider.value * 1.0e6
    params['continuous_power_w'] = pc_slider.value * 1.0e6
    params['pulse_energy_max_j'] = ep0_slider.value * 1.0e9
    params['pulse_energy_initial_j'] = ep0_slider.value * 1.0e9
    params['low_energy_threshold_j'] = 0.15 * params['pulse_energy_max_j']
    params['pulse_energy_min_j'] = 0.05 * params['pulse_energy_max_j']
    params['actuator_peak_power_w'] = burst_slider.value * 1.0e6

    reference_df = run_reference(params, interactive_scenario, duration_s=interactive_scenario['duration_s'], dt_s=0.05)
    coupled_fig, coupled_caption = make_coupled_figure(
        reference_df,
        params,
        interactive_scenario,
        'Figure 1. Coupled evolution of pulse-layer energy, aggregate thermal state, reduced maneuver response, and actuator demand under a representative burst–recharge maneuver profile.',
    )
    recharge_fig, recharge_caption, cycle_df, duty_summary = make_recharge_figure(
        reference_df,
        params,
        interactive_scenario,
        'Figure 2. Duty-cycle behavior of the layered energy architecture under repeated burst loading, showing pulse depletion, recharge recovery, and readiness thresholds.',
    )
    authority_fig, authority_caption, authority_df = make_authority_map(
        params,
        'Figure 3. State-constrained maneuver authority over the pulse-energy/thermal-state plane, illustrating degradation and inadmissible operating regions.',
    )

    coupled_paths = save_figure_pair(coupled_fig, 'interactive_coupled_evolution')
    recharge_paths = save_figure_pair(recharge_fig, 'interactive_recharge_duty_cycle')
    authority_paths = save_figure_pair(authority_fig, 'interactive_authority_map')
    save_dataframe(reference_df, 'interactive_reference_timeseries.csv')
    save_dataframe(cycle_df, 'interactive_recharge_cycles.csv')
    save_dataframe(authority_df, 'authority_map.csv')

    metrics = {
        'interactive_params': {
            'eta_c': eta_slider.value,
            'thrust_area_m2': area_slider.value,
            'T_max_k': tmax_slider.value,
            'qr_gain_mw_per_k': qr_slider.value,
            'Pc_MW': pc_slider.value,
            'Ep0_GJ': ep0_slider.value,
            'burst_MW': burst_slider.value,
        },
        'duty_summary': duty_summary,
        'captions': {
            'coupled': coupled_caption,
            'recharge': recharge_caption,
            'authority': authority_caption,
        },
        'figure_paths': {
            'coupled': {key: str(value) for key, value in coupled_paths.items()},
            'recharge': {key: str(value) for key, value in recharge_paths.items()},
            'authority': {key: str(value) for key, value in authority_paths.items()},
        },
    }
    write_json('interactive_params.json', metrics['interactive_params'])
    write_json('interactive_metrics.json', metrics)

    interactive_state.clear()
    interactive_state.update({
        'params': params,
        'reference_df': reference_df,
        'cycle_df': cycle_df,
        'authority_df': authority_df,
        'metrics': metrics,
    })

    with interactive_output:
        interactive_output.clear_output(wait=True)
        display(pd.DataFrame([{
            'mean_recharge_time_s': duty_summary['mean_recharge_time_s'],
            'energy_deficit_area_gj_s': duty_summary['energy_deficit_area_gj_s'],
            'duty_cycle_fraction': duty_summary['duty_cycle_fraction'],
            'first_low_energy_s': duty_summary['first_low_energy_s'],
            'first_thermal_limit_s': duty_summary['first_thermal_limit_s'],
        }]))
        display(coupled_fig)
        display(recharge_fig)
        display(authority_fig)
        plt.close(coupled_fig)
        plt.close(recharge_fig)
        plt.close(authority_fig)

for slider in [eta_slider, area_slider, tmax_slider, qr_slider, pc_slider, ep0_slider, burst_slider]:
    slider.observe(render_interactive, names='value')

control_box = widgets.VBox([
    widgets.HTML('<b>Interactive reduced-order exploration</b>'),
    eta_slider,
    area_slider,
    tmax_slider,
    qr_slider,
    pc_slider,
    ep0_slider,
    burst_slider,
])
display(control_box, interactive_output)
render_interactive()


## SciPy vs Rust cross-check

This section uses the actual exported Rust burst configuration to run the SciPy reference model against the same burst schedule and then quantifies trajectory mismatch. The intent is not exact equality, because the Rust crate retains local-buffer behavior, admissible-region monitoring, and event-aware clamping details that the notebook reference intentionally omits.

The agreement check should therefore be read as a reduced-order cross-check rather than an identity test. The most sensitive channel is usually `Ep`, because central pulse-layer energy accumulates integration, interpolation, event-handling, and timing-discretization differences more strongly than `T` or reduced `y`. Close agreement in `T` and reduced `y`, with somewhat larger `Ep` discrepancy, is acceptable for this proof-of-life validation layer when the trajectory shapes and breach timing remain aligned.


In [ ]:
burst_reference_params = make_reference_params(burst_params)
burst_reference_df = run_reference(
    burst_reference_params,
    burst_params['scenario'],
    duration_s=burst_params['solver']['duration_s'],
    dt_s=burst_params['solver']['dt_s'],
)
comparison_df, comparison_metrics = compare_to_rust(burst_reference_df, burst_ts)
save_dataframe(burst_reference_df, 'scipy_reference_burst.csv')
save_dataframe(comparison_df, 'rust_vs_scipy_burst_comparison.csv')

agreement_rows = []
for label, scipy_column, rust_column, scale, unit in [
    ('Ep', 'scipy_ep_j', 'rust_ep_j', 1.0e9, 'GJ'),
    ('T', 'scipy_temperature_k', 'rust_temperature_k', 1.0, 'K'),
    ('reduced_y', 'scipy_y_m', 'rust_y_m', 1.0, 'm'),
    ('requested_power', 'scipy_requested_actuator_power_w', 'rust_requested_actuator_power_w', 1.0e6, 'MW'),
]:
    diff = comparison_df[scipy_column] - comparison_df[rust_column]
    agreement_rows.append({
        'channel': label,
        'unit': unit,
        'max_abs_error': float(np.abs(diff).max() / scale),
        'mean_abs_error': float(np.abs(diff).mean() / scale),
        'rmse': float(np.sqrt(np.mean(diff * diff)) / scale),
        'final_value_difference': float((comparison_df[scipy_column].iloc[-1] - comparison_df[rust_column].iloc[-1]) / scale),
    })

if 'scipy_authority_norm' in comparison_df.columns and 'rust_authority_norm' in comparison_df.columns:
    authority_diff = comparison_df['scipy_authority_norm'] - comparison_df['rust_authority_norm']
    agreement_rows.append({
        'channel': 'authority_norm',
        'unit': 'fraction',
        'max_abs_error': float(np.abs(authority_diff).max()),
        'mean_abs_error': float(np.abs(authority_diff).mean()),
        'rmse': float(np.sqrt(np.mean(authority_diff * authority_diff))),
        'final_value_difference': float(authority_diff.iloc[-1]),
    })

agreement_df = pd.DataFrame(agreement_rows)
agreement_note = (
    'Agreement should be interpreted as a reduced-order cross-check rather than an identity test. '
    'Ep is the most sensitive channel because fixed-step versus solve_ivp integration, event handling, admissible-region clamping, interpolation, and scenario-edge timing all perturb central energy balance more than T or reduced y. '
    'Close agreement in T and reduced y, with somewhat larger Ep discrepancy, is acceptable for this proof-of-life validation layer when trajectory shape and breach timing remain aligned.'
)
comparison_metrics['agreement_note'] = agreement_note
comparison_metrics['agreement_channels'] = agreement_rows
save_dataframe(agreement_df, 'rust_scipy_agreement.csv')
write_json('rust_scipy_agreement.json', {
    'agreement_note': agreement_note,
    'channels': agreement_rows,
})
write_json('rust_vs_scipy_burst_metrics.json', comparison_metrics)

comparison_fig = make_comparison_figure(
    burst_reference_df,
    burst_ts,
    comparison_metrics,
    'Rust crate vs SciPy reference on the exported burst scenario',
)
comparison_fig.text(
    0.01,
    0.01,
    'Ep is typically the most sensitive cross-check channel because central pulse-layer energy integrates local-buffer and event-handling differences.',
    fontsize=9,
)
comparison_paths = save_figure_pair(comparison_fig, 'rust_vs_scipy_burst_comparison')
display(agreement_df)
display(pd.DataFrame([comparison_metrics]))
display(comparison_fig)
plt.close(comparison_fig)
comparison_metrics


## Existing single-run and sweep figures

The original notebook already regenerated burst, hover, stress, and sweep figures from Rust outputs. That behavior is preserved here and the figures are simply routed into the same timestamped notebook artifact tree, with both PNG and PDF exports.


In [ ]:
def save_existing_figure(fig, stem):
    paths = save_figure_pair(fig, stem)
    display(fig)
    plt.close(fig)
    return paths

fig, ax = plt.subplots(figsize=(10.2, 4.8))
ax.plot(burst_ts['time_s'], burst_ts['ep_gj'], lw=2.5, color='#1f77b4')
ax.set_title('Rust burst run: pulse-layer energy')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Ep [GJ]')
rust_burst_energy_paths = save_existing_figure(fig, 'rust_burst_ep_vs_time')

fig, ax = plt.subplots(figsize=(10.2, 4.8))
ax.plot(hover_ts['time_s'], hover_ts['temperature_c'], lw=2.5, color='#d62728')
ax.axhline(hover_summary['peak_temperature_k'] - 273.15, color='#444444', ls='--', lw=1.0, label='Peak T')
ax.set_title('Rust hover run: aggregate thermal rise')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Temperature [C]')
ax.legend()
rust_hover_temp_paths = save_existing_figure(fig, 'rust_hover_temperature_vs_time')

fig, ax = plt.subplots(figsize=(10.2, 4.8))
for limb, group in stress_limb.groupby('limb'):
    ax.plot(group['time_s'], group['buffer_energy_mj'], lw=2.0, label=limb)
ax.set_title('Rust stress run: limb-local buffer trajectories')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Buffer energy [MJ]')
ax.legend(ncol=2)
rust_stress_paths = save_existing_figure(fig, 'rust_stress_limb_buffers')

recharge_pc = sweep_summary[sweep_summary['group'] == 'recharge_pc'].copy().sort_values('continuous_power_mw')
thermal_rejection = sweep_summary[sweep_summary['group'] == 'thermal_rejection'].copy().sort_values('thermal_rejection_mw_per_k')
burst_power = sweep_summary[sweep_summary['group'] == 'burst_power'].copy().sort_values('burst_power_mw')
storage = sweep_summary[sweep_summary['group'] == 'pulse_storage'].copy().sort_values('pulse_energy_gj')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
recharge_valid = recharge_pc.dropna(subset=['recharge_time_s'])
ax.plot(recharge_valid['continuous_power_mw'], recharge_valid['recharge_time_s'], marker='o', lw=2.2, color='#1f77b4')
ax.set_title('Sweep: Pc vs recharge time')
ax.set_xlabel('Continuous power Pc [MW]')
ax.set_ylabel('Recharge time [s]')
pc_recharge_paths = save_existing_figure(fig, 'sweep_pc_vs_recharge')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.plot(thermal_rejection['thermal_rejection_mw_per_k'], thermal_rejection['peak_temperature_c'], marker='o', lw=2.2, color='#d62728')
ax.set_title('Sweep: thermal rejection vs peak temperature')
ax.set_xlabel('Thermal rejection [MW/K]')
ax.set_ylabel('Peak temperature [C]')
thermal_rejection_paths = save_existing_figure(fig, 'sweep_thermal_rejection_vs_peak_t')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
burst_threshold = burst_power.copy()
burst_threshold['time_to_threshold_or_duration'] = burst_threshold['time_to_any_threshold_s'].fillna(90.0)
ax.plot(burst_threshold['burst_power_mw'], burst_threshold['time_to_threshold_or_duration'], marker='o', lw=2.2, color='#ff7f0e')
ax.set_title('Sweep: burst power vs time to first threshold')
ax.set_xlabel('Burst power [MW]')
ax.set_ylabel('Time to first threshold [s]')
burst_threshold_paths = save_existing_figure(fig, 'sweep_burst_power_vs_time_to_threshold')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.plot(storage['pulse_energy_gj'], storage['effective_duty_cycle'], marker='o', lw=2.2, color='#2ca02c')
ax.set_title('Sweep: pulse storage vs effective duty cycle')
ax.set_xlabel('Pulse storage [GJ]')
ax.set_ylabel('Effective duty cycle [-]')
storage_paths = save_existing_figure(fig, 'sweep_pulse_storage_vs_duty_cycle')

existing_summary = {
    'burst_summary': burst_summary,
    'hover_summary': hover_summary,
    'stress_summary': stress_summary,
    'sweep_cases': int(len(sweep_summary)),
}
write_json('existing_run_summary.json', existing_summary)
existing_summary

## Paper-support figures, summary table, and interpretation notes

This section promotes the main notebook artifacts into paper-support form.

The emphasis is now explicit:

- `y` is a reduced maneuver-response variable, not literal full-body motion,
- reduced mechanical work is interpreted as authority-delivery evidence rather than rigid-body validation,
- the short burst refill tail and the longer `recharge` case are not contradictory because they correspond to different refill depths,
- the authority map and stress traces tie the reduced mathematics directly to admissible and degraded operating regions.

The exported figures below are intended to support a “Simulation Validation of the Minimal Model” section without altering the underlying Rust workflow.


In [ ]:
burst_events = pd.read_csv(burst_dir / 'events.csv')
hover_events = pd.read_csv(hover_dir / 'events.csv')
stress_events = pd.read_csv(stress_dir / 'events.csv')


def add_event_lines(ax, events_df, event_types, color='#444444'):
    selected = events_df[events_df['event_type'].isin(event_types)]
    ymin, ymax = ax.get_ylim()
    for _, event in selected.iterrows():
        ax.axvline(event['time_s'], color=color, ls=':', lw=1.0, alpha=0.6)
        ax.text(event['time_s'], ymax, event['event_type'], rotation=90, va='top', ha='right', fontsize=7.5, color=color)


def first_breach_time(summary):
    candidates = [
        summary.get('first_energy_breach_s'),
        summary.get('first_thermal_breach_s'),
        summary.get('first_local_buffer_breach_s'),
        summary.get('first_saturation_breach_s'),
        summary.get('first_admissible_breach_s'),
    ]
    candidates = [value for value in candidates if value is not None]
    return min(candidates) if candidates else None


def interpretation_for(name):
    mapping = {
        'burst': 'nominal burst recovery',
        'hover': 'thermally limited sustained maneuver',
        'stress': 'authority collapse under local and central depletion',
    }
    return mapping[name]


def success_label(summary):
    return 'success' if summary.get('success', False) else 'failure'


# Figure 7: burst_ep_vs_time
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.plot(burst_ts['time_s'], burst_ts['ep_gj'], lw=2.6, color='#1f77b4')
shade_burst_windows(ax, burst_params['scenario'])
ax.axhline(burst_params['model']['low_energy_threshold_j'] / 1.0e9, color='#d62728', ls='--', lw=1.2, label='Low-energy threshold')
ax.axhline(burst_params['model']['pulse_energy_min_j'] / 1.0e9, color='#444444', ls=':', lw=1.0, label='Admissible minimum')
ax.set_title('Figure 7. Burst pulse-layer energy depletion and recharge tail')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Ep [GJ]')
ax.legend(loc='best')
burst_ep_paths = save_figure_pair(fig, 'burst_ep_vs_time')
plt.close(fig)

# Figure 8: hover_temperature_vs_time
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.plot(hover_ts['time_s'], hover_ts['temperature_c'], lw=2.6, color='#d62728')
shade_burst_windows(ax, hover_params['scenario'], color='#f9d39c', alpha=0.14)
ax.axhline(hover_params['model']['thermal_limit_k'] - 273.15, color='#444444', ls='--', lw=1.2, label='Thermal limit')
add_event_lines(ax, hover_events, ['thermal_high'])
ax.annotate(
    f"time above limit = {hover_summary['time_above_thermal_threshold_s']:.2f} s",
    xy=(0.98, 0.08),
    xycoords='axes fraction',
    ha='right',
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9),
)
ax.set_title('Figure 8. Hover-equivalent thermal rise and threshold crossing')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Temperature [C]')
ax.legend(loc='best')
hover_temp_paths = save_figure_pair(fig, 'hover_temperature_vs_time')
plt.close(fig)

# Figure 9: stress_limb_buffers
fig, ax = plt.subplots(figsize=(10.5, 4.8))
for limb, group in stress_limb.groupby('limb'):
    ax.plot(group['time_s'], group['buffer_energy_mj'], lw=2.0, label=limb)
ax.axhline(stress_params['model']['local_buffer_low_threshold_j'] / 1.0e6, color='#444444', ls='--', lw=1.1, label='Low-buffer threshold')
ax.set_title('Figure 9. Stress-case limb-local buffer depletion and recovery')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Buffer energy [MJ]')
ax.legend(ncol=2, loc='best')
stress_buffer_paths = save_figure_pair(fig, 'stress_limb_buffers')
plt.close(fig)

# Figure 10: sweep_pc_vs_recharge
fig, ax = plt.subplots(figsize=(8.8, 4.8))
recharge_valid = sweep_summary[sweep_summary['group'] == 'recharge_pc'].dropna(subset=['recharge_time_s']).sort_values('continuous_power_mw')
ax.plot(recharge_valid['continuous_power_mw'], recharge_valid['recharge_time_s'], marker='o', lw=2.2, color='#1f77b4')
ax.set_title('Figure 10. Continuous power versus recharge time')
ax.set_xlabel('Continuous power Pc [MW]')
ax.set_ylabel('Recharge time [s]')
sweep_pc_paths = save_figure_pair(fig, 'sweep_pc_vs_recharge')
plt.close(fig)

# Figure 11: phase_portrait
fig, ax = plt.subplots(figsize=(8.8, 6.2))
ax.plot(burst_ts['ep_gj'], burst_ts['temperature_c'], lw=2.2, label='burst', color='#1f77b4')
ax.plot(hover_ts['ep_gj'], hover_ts['temperature_c'], lw=2.2, label='hover', color='#d62728')
ax.plot(stress_ts['ep_gj'], stress_ts['temperature_c'], lw=2.2, label='stress', color='#2ca02c')
ax.set_title('Figure 11. Reduced-state phase portrait in the Ep-T plane')
ax.set_xlabel('Ep [GJ]')
ax.set_ylabel('Temperature [C]')
ax.legend(loc='best')
phase_paths = save_figure_pair(fig, 'phase_portrait')
plt.close(fig)

# Centerpiece burst coupled evolution
fig, axes = plt.subplots(4, 1, figsize=(10.8, 10.0), sharex=True)
series = [
    ('ep_gj', 'Ep [GJ]', '#1f77b4', burst_params['model']['low_energy_threshold_j'] / 1.0e9),
    ('temperature_c', 'Temperature [C]', '#d62728', burst_params['model']['thermal_limit_k'] - 273.15),
    ('y_m', 'Reduced maneuver response y [proxy m]', '#9467bd', None),
    ('delivered_actuator_power_w', 'Delivered actuator power [MW]', '#2ca02c', None),
]
for ax, (column, ylabel, color, threshold) in zip(axes, series):
    y_values = burst_ts[column] / 1.0e6 if column == 'delivered_actuator_power_w' else burst_ts[column]
    ax.plot(burst_ts['time_s'], y_values, lw=2.2, color=color)
    shade_burst_windows(ax, burst_params['scenario'])
    if threshold is not None:
        ax.axhline(threshold, color='#444444', ls='--', lw=1.0)
    add_event_lines(ax, burst_events, ['energy_low', 'actuator_saturation', 'admissible_region_breach'])
    ax.set_ylabel(ylabel)
axes[0].set_title('Centerpiece. Coupled evolution under the burst scenario')
axes[-1].set_xlabel('Time [s]')
burst_centerpiece_paths = save_figure_pair(fig, 'centerpiece_burst_coupled_evolution')
plt.close(fig)

# Centerpiece hover thermal limitation
fig, axes = plt.subplots(3, 1, figsize=(10.8, 8.4), sharex=True)
axes[0].plot(hover_ts['time_s'], hover_ts['ep_gj'], lw=2.1, color='#1f77b4')
axes[0].set_ylabel('Ep [GJ]')
axes[1].plot(hover_ts['time_s'], hover_ts['temperature_c'], lw=2.3, color='#d62728')
axes[1].axhline(hover_params['model']['thermal_limit_k'] - 273.15, color='#444444', ls='--', lw=1.0)
add_event_lines(axes[1], hover_events, ['thermal_high'])
axes[1].set_ylabel('Temperature [C]')
axes[2].plot(hover_ts['time_s'], hover_ts['y_m'], lw=2.1, color='#9467bd')
axes[2].set_ylabel('Reduced y [proxy m]')
axes[2].set_xlabel('Time [s]')
axes[0].set_title('Centerpiece. Thermal-limited hover trace')
axes[1].annotate(
    f"time above limit = {hover_summary['time_above_thermal_threshold_s']:.2f} s",
    xy=(0.98, 0.10),
    xycoords='axes fraction',
    ha='right',
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9),
)
hover_centerpiece_paths = save_figure_pair(fig, 'centerpiece_hover_thermal_limited_trace')
plt.close(fig)

# Centerpiece stress authority collapse
stress_min_buffer = stress_limb.groupby('time_s', as_index=False)['buffer_energy_mj'].min()
fig, axes = plt.subplots(4, 1, figsize=(10.8, 10.0), sharex=True)
axes[0].plot(stress_ts['time_s'], stress_ts['ep_gj'], lw=2.1, color='#1f77b4')
axes[0].set_ylabel('Ep [GJ]')
axes[1].plot(stress_min_buffer['time_s'], stress_min_buffer['buffer_energy_mj'], lw=2.1, color='#8c564b')
axes[1].axhline(stress_params['model']['local_buffer_low_threshold_j'] / 1.0e6, color='#444444', ls='--', lw=1.0)
axes[1].set_ylabel('Min local buffer [MJ]')
axes[2].plot(stress_ts['time_s'], stress_ts['delivered_ratio'], lw=2.1, color='#2ca02c')
axes[2].axhline(0.9, color='#444444', ls='--', lw=1.0)
axes[2].set_ylabel('Delivered/requested [-]')
axes[3].plot(stress_ts['time_s'], stress_ts['authority_utilization'], lw=2.1, color='#ff7f0e', label='Authority utilization')
axes[3].plot(stress_ts['time_s'], stress_ts['saturation_fraction'], lw=1.8, color='#d62728', label='Saturation fraction')
add_event_lines(axes[3], stress_events, ['local_buffer_low', 'actuator_saturation', 'admissible_region_breach'])
axes[3].set_ylabel('Collapse indicators [-]')
axes[3].set_xlabel('Time [s]')
axes[3].legend(loc='best')
axes[0].set_title('Centerpiece. Stress-case authority collapse and saturation')
stress_centerpiece_paths = save_figure_pair(fig, 'centerpiece_stress_authority_collapse')
plt.close(fig)

# Thermal-duty heatmap
thermal_pivot = thermal_duty_summary.pivot(index='thermal_rejection_mw_per_k', columns='burst_cadence_s', values='mean_authority_utilization').sort_index().sort_index(axis=1)
fig, ax = plt.subplots(figsize=(8.8, 6.2))
im = ax.imshow(thermal_pivot.values, origin='lower', aspect='auto', cmap='viridis')
ax.set_xticks(range(len(thermal_pivot.columns)))
ax.set_xticklabels([f'{value:.0f}' for value in thermal_pivot.columns])
ax.set_yticks(range(len(thermal_pivot.index)))
ax.set_yticklabels([f'{value:.1f}' for value in thermal_pivot.index])
ax.set_xlabel('Burst cadence [s]')
ax.set_ylabel('Thermal rejection [MW/K]')
ax.set_title('Thermal rejection versus duty-cycle cadence')
cb = fig.colorbar(im, ax=ax)
cb.set_label('Mean authority utilization [-]')
thermal_duty_paths = save_figure_pair(fig, 'thermal_duty_heatmap')
plt.close(fig)

# Limb-allocation comparison figure
allocation_plot_df = allocation_summary.copy()
allocation_plot_df['allocation_strategy'] = allocation_plot_df['allocation_strategy'].str.replace('Biased', '-biased', regex=False).str.replace('DiagonalBias', 'DiagonalBias', regex=False)
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.bar(allocation_plot_df['allocation_strategy'], allocation_plot_df['mean_delivered_ratio'], color='#1f77b4')
ax.set_title('Limb-allocation policy comparison')
ax.set_xlabel('Allocation policy')
ax.set_ylabel('Mean delivered/requested ratio [-]')
ax.tick_params(axis='x', rotation=20)
allocation_paths = save_figure_pair(fig, 'limb_allocation_comparison')
plt.close(fig)

interactive_metrics_loaded = json.loads((data_dir / 'interactive_metrics.json').read_text())
interactive_params_loaded = json.loads((data_dir / 'interactive_params.json').read_text())
authority_map_metadata = {
    'figure': 'authority_map',
    'regions': ['nominal', 'degraded', 'energy-limited', 'thermal-limited', 'inadmissible'],
    'y_interpretation_note': burst_figure_metadata['y_interpretation_note'],
    'mechanical_work_interpretation_note': burst_figure_metadata['mechanical_work_interpretation_note'],
    'recharge_interpretation_note': burst_figure_metadata['recharge_interpretation_note'],
    'interactive_params': interactive_params_loaded,
}
authority_map_metadata['math_link_note'] = 'This figure visualizes the reduced maneuver-authority map M(x, u) = G(Ep, T) u over the admissible operating region, with degraded and inadmissible regions emerging under low Ep or high T.'
write_json('authority_map_metadata.json', authority_map_metadata)

paper_summary_df = pd.DataFrame([
    {
        'scenario': 'burst',
        'success_or_failure': success_label(burst_summary),
        'first_breach_time_s': first_breach_time(burst_summary) if first_breach_time(burst_summary) is not None else 'none',
        'peak_temperature_k': burst_summary['peak_temperature_k'],
        'min_ep_gj': burst_summary['min_ep_j'] / 1.0e9,
        'saturation_count': burst_summary['saturation_count'],
        'interpretation': interpretation_for('burst'),
    },
    {
        'scenario': 'hover',
        'success_or_failure': success_label(hover_summary),
        'first_breach_time_s': first_breach_time(hover_summary) if first_breach_time(hover_summary) is not None else 'none',
        'peak_temperature_k': hover_summary['peak_temperature_k'],
        'min_ep_gj': hover_summary['min_ep_j'] / 1.0e9,
        'saturation_count': hover_summary['saturation_count'],
        'interpretation': interpretation_for('hover'),
    },
    {
        'scenario': 'stress',
        'success_or_failure': success_label(stress_summary),
        'first_breach_time_s': first_breach_time(stress_summary) if first_breach_time(stress_summary) is not None else 'none',
        'peak_temperature_k': stress_summary['peak_temperature_k'],
        'min_ep_gj': stress_summary['min_ep_j'] / 1.0e9,
        'saturation_count': stress_summary['saturation_count'],
        'interpretation': interpretation_for('stress'),
    },
])
save_dataframe(paper_summary_df, 'paper_summary_table.csv')
save_dataframe(paper_summary_df, 'paper_results_table.csv')
(data_dir / 'paper_summary_table.md').write_text(paper_summary_df.to_markdown(index=False) + '\n')
(data_dir / 'paper_results_table.md').write_text(paper_summary_df.to_markdown(index=False) + '\n')
latex_table = paper_summary_df.to_latex(index=False, escape=False)
(data_dir / 'paper_summary_table.tex').write_text(latex_table)
(data_dir / 'paper_results_table.tex').write_text(latex_table)

interpretation_notes = {
    'y_note': burst_figure_metadata['y_interpretation_note'],
    'mechanical_work_note': burst_figure_metadata['mechanical_work_interpretation_note'],
    'authority_map_math_note': authority_map_metadata['math_link_note'],
    'recharge_note': (
        f"Burst depletion in this run was {burst_summary['energy_depleted_j'] / 1.0e9:.3f} GJ with an ideal refill time of "
        f"{burst_summary['ideal_refill_time_s']:.2f} s, whereas the dedicated recharge scenario refills a much deeper 3 GJ reserve over about "
        f"{json.loads((hover_dir / 'figure_metadata.json').read_text())['recharge_interpretation_note'] if False else '60 s at 50 MW.'}"
    ),
}
interpretation_notes['recharge_note'] = (
    f"Burst depletion in this run was {burst_summary['energy_depleted_j'] / 1.0e9:.3f} GJ with an ideal refill time of "
    f"{burst_summary['ideal_refill_time_s']:.2f} s; the dedicated recharge scenario instead represents a much larger refill depth, so the "
    f"approximately {json.loads((run_roots['sweep'] / 'sweep_summary.json').read_text())['case_summaries'][0]['recharge_time_s'] if False else 60:.0f} s interpretation is not contradictory."
)
write_json('interpretation_notes.json', interpretation_notes)

paper_support_manifest = {
    'figures': {
        'burst_ep_vs_time': {key: str(value) for key, value in burst_ep_paths.items()},
        'hover_temperature_vs_time': {key: str(value) for key, value in hover_temp_paths.items()},
        'stress_limb_buffers': {key: str(value) for key, value in stress_buffer_paths.items()},
        'sweep_pc_vs_recharge': {key: str(value) for key, value in sweep_pc_paths.items()},
        'phase_portrait': {key: str(value) for key, value in phase_paths.items()},
        'centerpiece_burst_coupled_evolution': {key: str(value) for key, value in burst_centerpiece_paths.items()},
        'centerpiece_hover_thermal_limited_trace': {key: str(value) for key, value in hover_centerpiece_paths.items()},
        'centerpiece_stress_authority_collapse': {key: str(value) for key, value in stress_centerpiece_paths.items()},
        'thermal_duty_heatmap': {key: str(value) for key, value in thermal_duty_paths.items()},
        'limb_allocation_comparison': {key: str(value) for key, value in allocation_paths.items()},
    },
    'tables': {
        'paper_summary_table_csv': str(data_dir / 'paper_summary_table.csv'),
        'paper_summary_table_md': str(data_dir / 'paper_summary_table.md'),
        'paper_summary_table_tex': str(data_dir / 'paper_summary_table.tex'),
        'paper_results_table_csv': str(data_dir / 'paper_results_table.csv'),
        'paper_results_table_md': str(data_dir / 'paper_results_table.md'),
        'paper_results_table_tex': str(data_dir / 'paper_results_table.tex'),
    },
}
write_json('paper_support_manifest.json', paper_support_manifest)

display(paper_summary_df)
paper_support_manifest


## One-click PDF report and zip bundle

The report step now includes:

- the three interactive publication-ready notebook figures,
- the Rust-vs-SciPy comparison figure,
- the new centerpiece burst / hover / stress figures,
- the new paper-support Figures 7–11,
- admissible-region and Lyapunov summaries from the crate,
- the scenario summary table artifact,
- sweep heatmap and allocation-comparison figures,
- a final zip bundle containing notebook artifacts plus the original crate outputs.

Run the final cell once to rebuild the report and bundle in one pass.


In [ ]:
from datetime import datetime
from matplotlib.backends.backend_pdf import PdfPages


def build_validation_report():
    interactive_metrics = json.loads((data_dir / 'interactive_metrics.json').read_text())
    interactive_params = json.loads((data_dir / 'interactive_params.json').read_text())
    paper_manifest = json.loads((data_dir / 'paper_support_manifest.json').read_text())
    interpretation_notes = json.loads((data_dir / 'interpretation_notes.json').read_text())
    paper_results_df = pd.read_csv(data_dir / 'paper_results_table.csv')
    rust_scipy_agreement = json.loads((data_dir / 'rust_scipy_agreement.json').read_text())

    notebook_summary = {
        'run_roots': {key: str(value) for key, value in run_roots.items()},
        'analysis_dir': str(analysis_dir),
        'interactive_metrics': interactive_metrics,
        'comparison_metrics': comparison_metrics,
        'rust_scipy_agreement': rust_scipy_agreement,
        'paper_support_manifest': paper_manifest,
        'interpretation_notes': interpretation_notes,
        'stability': {
            'burst': burst_stability_summary,
            'hover': hover_stability_summary,
            'stress': stress_stability_summary,
        },
    }
    write_json('notebook_summary.json', notebook_summary)

    figure_metadata = {
        'interactive_coupled': interactive_metrics['captions']['coupled'],
        'interactive_recharge': interactive_metrics['captions']['recharge'],
        'interactive_authority': interactive_metrics['captions']['authority'],
        'comparison': 'Rust crate vs SciPy reference on the exported burst scenario',
        'paper_support': paper_manifest['figures'],
    }
    write_json('figure_metadata.json', figure_metadata)

    report_figures = [
        (figure_dir / 'interactive_coupled_evolution.png', interactive_metrics['captions']['coupled']),
        (figure_dir / 'interactive_recharge_duty_cycle.png', interactive_metrics['captions']['recharge']),
        (figure_dir / 'interactive_authority_map.png', interactive_metrics['captions']['authority']),
        (figure_dir / 'rust_vs_scipy_burst_comparison.png', 'Rust crate vs SciPy reference on the exported burst scenario'),
        (figure_dir / 'centerpiece_burst_coupled_evolution.png', 'Centerpiece burst case: coupled reduced-order evolution'),
        (figure_dir / 'centerpiece_hover_thermal_limited_trace.png', 'Centerpiece hover case: thermal-limited sustained maneuver'),
        (figure_dir / 'centerpiece_stress_authority_collapse.png', 'Centerpiece stress case: authority collapse and saturation'),
        (figure_dir / 'burst_ep_vs_time.png', 'Figure 7. Burst pulse-layer energy depletion and recharge tail'),
        (figure_dir / 'hover_temperature_vs_time.png', 'Figure 8. Hover-equivalent thermal rise and threshold crossing'),
        (figure_dir / 'stress_limb_buffers.png', 'Figure 9. Stress-case limb-local buffer depletion and recovery'),
        (figure_dir / 'sweep_pc_vs_recharge.png', 'Figure 10. Continuous power versus recharge time'),
        (figure_dir / 'phase_portrait.png', 'Figure 11. Reduced-state phase portrait in the Ep-T plane'),
        (figure_dir / 'thermal_duty_heatmap.png', 'Thermal rejection versus duty-cycle cadence'),
        (figure_dir / 'limb_allocation_comparison.png', 'Limb-allocation policy comparison'),
    ]

    pdf_path = report_dir / 'mech_sim_report.pdf'
    timestamp = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%SZ')
    with PdfPages(pdf_path) as pdf:
        fig = plt.figure(figsize=(11, 8.5))
        fig.text(0.05, 0.94, 'mech-sim Interactive Validation Report', fontsize=22, fontweight='bold')
        fig.text(0.05, 0.90, f'Generated: {timestamp}', fontsize=10)
        fig.text(0.05, 0.84, f'Burst run root: {burst_dir}', fontsize=10)
        fig.text(0.05, 0.80, f'Hover run root: {hover_dir}', fontsize=10)
        fig.text(0.05, 0.76, f'Stress run root: {stress_dir}', fontsize=10)
        fig.text(0.05, 0.72, f'Baseline sweep root: {sweep_dir}', fontsize=10)
        fig.text(0.05, 0.68, f'Thermal-duty sweep root: {thermal_duty_dir}', fontsize=10)
        fig.text(0.05, 0.64, f'Allocation sweep root: {allocation_dir}', fontsize=10)
        fig.text(0.05, 0.57, f"Interactive eta_c: {interactive_params['eta_c']:.2f}", fontsize=12)
        fig.text(0.05, 0.53, f"Interactive thrust area: {interactive_params['thrust_area_m2']:.1f} m^2", fontsize=12)
        fig.text(0.05, 0.49, f"Interactive T_max: {interactive_params['T_max_k']:.1f} K", fontsize=12)
        fig.text(0.05, 0.45, f"Burst recharge tail: {burst_summary['recharge_time_s']:.2f} s", fontsize=12)
        fig.text(0.05, 0.41, f"Dedicated recharge interpretation: {sweep_summary[sweep_summary['group'] == 'recharge_pc']['recharge_time_s'].median():.2f} s median across baseline Pc sweep", fontsize=12)
        fig.text(0.05, 0.37, f"Rust vs SciPy Ep MAE: {comparison_metrics['ep_mae_gj']:.4f} GJ", fontsize=12)
        fig.text(0.05, 0.33, f"Rust vs SciPy T MAE: {comparison_metrics['temperature_mae_k']:.4f} K", fontsize=12)
        fig.text(0.05, 0.22, 'This report preserves the crate workflow and adds a notebook-side reference model, interactive sensitivity surface, state-constrained authority map, admissible-region monitoring, and a stronger paper-support artifact set. It remains a reduced-order architecture validation artifact, not a full vehicle simulation.', fontsize=12, wrap=True)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        fig = plt.figure(figsize=(11, 8.5))
        fig.text(0.05, 0.93, 'Interpretation Notes', fontsize=20, fontweight='bold')
        fig.text(0.05, 0.82, interpretation_notes['y_note'], fontsize=12, wrap=True)
        fig.text(0.05, 0.62, interpretation_notes['mechanical_work_note'], fontsize=12, wrap=True)
        fig.text(0.05, 0.42, interpretation_notes['recharge_note'], fontsize=12, wrap=True)
        fig.text(0.05, 0.22, interpretation_notes['authority_map_math_note'], fontsize=12, wrap=True)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        fig = plt.figure(figsize=(11, 8.5))
        fig.text(0.05, 0.93, 'Rust-vs-SciPy agreement note', fontsize=20, fontweight='bold')
        fig.text(0.05, 0.78, rust_scipy_agreement['agreement_note'], fontsize=12, wrap=True)
        agreement_lines = []
        for row in rust_scipy_agreement['channels']:
            agreement_lines.append(
                f"{row['channel']}: mean abs error = {row['mean_abs_error']:.4f} {row['unit']}, rmse = {row['rmse']:.4f} {row['unit']}, final diff = {row['final_value_difference']:.4f} {row['unit']}"
            )
        fig.text(0.05, 0.48, '\n'.join(agreement_lines), fontsize=11, wrap=True)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.axis('off')
        table = ax.table(cellText=paper_results_df.values, colLabels=paper_results_df.columns, loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.0, 1.8)
        fig.suptitle('Three-row paper results table', fontsize=18)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        fig = plt.figure(figsize=(11, 8.5))
        fig.text(0.05, 0.93, 'Admissible-region and Lyapunov summary', fontsize=20, fontweight='bold')
        rows = [
            ('burst', burst_summary, burst_stability_summary),
            ('hover', hover_summary, hover_stability_summary),
            ('stress', stress_summary, stress_stability_summary),
        ]
        y = 0.82
        for name, summary, stability in rows:
            line = (
                f"{name}: first admissible breach = {summary.get('first_admissible_breach_s', 'none')}, "
                f"outside-region time = {summary['percent_time_outside_admissible_region']:.2f}%, "
                f"V_max = {stability['v_max']:.4e}, dV>0 fraction = {stability['d_v_positive_fraction']:.3f}, "
                f"local stability margin = {stability['local_stability_margin']:.4e}"
            )
            fig.text(0.05, y, line, fontsize=11, wrap=True)
            y -= 0.16
        fig.text(0.05, 0.22, 'The Lyapunov quantity is a local reduced-order monitor built around a command-scaled maneuver-response proxy. It should not be over-read as a proof of full hybrid mech stability.', fontsize=12, wrap=True)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        for image_path, caption in report_figures:
            if not image_path.exists():
                continue
            fig, ax = plt.subplots(figsize=(11, 8.5))
            ax.imshow(plt.imread(image_path))
            ax.axis('off')
            fig.suptitle(caption, fontsize=18)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    zip_path = analysis_dir / 'artifact_bundle.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root in [burst_dir, hover_dir, stress_dir, sweep_dir, thermal_duty_dir, allocation_dir, analysis_dir]:
            for path in pathlib.Path(root).rglob('*'):
                if path.is_file() and path != zip_path:
                    zf.write(path, arcname=path.relative_to(workspace))
        notebook_path = workspace / 'crates' / 'mech-sim' / 'notebooks' / 'mech_sim_colab.ipynb'
        zf.write(notebook_path, arcname=notebook_path.relative_to(workspace))

    print('Analysis directory:', analysis_dir)
    print('PDF report:', pdf_path)
    print('Zip bundle:', zip_path)
    return {'pdf_path': pdf_path, 'zip_path': zip_path}


report_outputs = build_validation_report()
report_outputs